# Clase 2 — Errores comunes y patrones de prompting

Antes de aprender técnicas avanzadas, vale la pena aprender a **diagnosticar** por qué un prompt no funciona. La mayoría de los problemas con los modelos de lenguaje se reducen a cinco errores recurrentes — y todos tienen solución.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Los cinco anti-patrones más frecuentes |
| 3 | Herramienta de diagnóstico |
| 4 | Reformulación sistemática |
| 5 | Actividad: corregir prompts rotos |

---
## 1. Configuración del entorno

Este notebook usa el mismo wrapper de la clase anterior. Si ya configuraste tu entorno, solo cambiá `BACKEND` y continuá.

**Cómo guardar tu API key de Gemini (si es la primera vez):**
1. Obtené tu key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. En tu terminal, desde la carpeta del proyecto:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. El archivo `.env` queda en tu máquina y nunca se sube al repositorio.

Si preferís no crear el archivo, la celda siguiente te pide la clave de forma interactiva.

In [4]:
import os
import getpass

BACKEND = "ollama"          # "gemini", "ollama", "local"
GEMINI_MODEL = "gemma-4-26b-a4b-it"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini (no se muestra): ")

print(f"Backend: {BACKEND}")

Backend: ollama


In [5]:
# ─── Inicializar cliente y wrapper ────────────────────────────────────────────

if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

elif BACKEND == "ollama":
    import ollama
    
    OLLAMA_MODEL = "llama3.2"  # Modelo que tenés descargado
    
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    print("Descargando modelo local...")

    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    print("Inicializando modelo local...")
    _llm_local = Llama(
        model_path=ruta_modelo,
        n_ctx=2048,
        n_gpu_layers=0,
        verbose=False
    )
    print("✅ Modelo local listo (usando CPU).")


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    
    elif BACKEND == "ollama":
        r = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            system=system_prompt,
            stream=False,
            options={
                "temperature": temperature,
                "num_predict": max_tokens,
            }
        )
        return r['response'].strip()

    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: llama3.2
Disponible. ¿En qué puedo ayudarte?


---
## 2. Los cinco anti-patrones más frecuentes

Un **anti-patrón** es una forma de escribir prompts que parece razonable pero produce resultados pobres de forma predecible.

| # | Anti-patrón | Síntoma | Causa raíz |
|---|---|---|---|
| 1 | **Instrucción vaga** | Respuesta genérica o demasiado larga | No se especificó la tarea ni el alcance |
| 2 | **Sin rol** | Tono y nivel inapropiados para el destinatario | El modelo elige el perfil por defecto |
| 3 | **Sin formato** | Texto en prosa cuando necesitás lista, o viceversa | El modelo estructura a su criterio |
| 4 | **Múltiples tareas en un prompt** | Solo responde la primera, ignora el resto | Ambigüedad sobre prioridad |
| 5 | **Negaciones sin alternativa** | Cumple la negación de forma inesperada | El modelo se enfoca en lo que *no* debe hacer |

> 💡 Los anti-patrones 1, 2 y 3 son los más comunes entre quienes recién empiezan. El 4 y el 5 aparecen cuando ya se tiene algo de experiencia pero no se estructura bien la complejidad.

---
## 3. Herramienta de diagnóstico

Antes de reescribir un prompt que no funcionó, conviene hacerse cuatro preguntas rápidas:

In [7]:
def diagnosticar_prompt(prompt):
    """Imprime un diagnóstico rápido del prompt ingresado."""

    print("─" * 55)
    print("DIAGNÓSTICO DEL PROMPT")
    print("─" * 55)
    print(f"Texto: {prompt[:80]}{'...' if len(prompt) > 80 else ''}")
    print()

    # Indicadores simples basados en el texto
    tiene_rol     = any(p in prompt.lower() for p in ["sos ", "eres ", "actuá ", "actúa "])
    tiene_formato = any(p in prompt.lower() for p in ["lista", "tabla", "puntos", "bullets", "formato", "columnas"])
    es_corto      = len(prompt.split()) < 8
    tiene_no      = prompt.lower().startswith("no ") or " no " in prompt.lower()[:30]
    tiene_y_ademas = prompt.count(".") > 2 or " y además" in prompt.lower() or " también" in prompt.lower()

    checks = [
        ("Tiene rol explícito",        tiene_rol,         "Agregá 'Sos un...' al comienzo"),
        ("Especifica el formato",       tiene_formato,     "Indicá cómo querés la respuesta: lista, tabla, párrafo..."),
        ("Instrucción suficientemente larga", not es_corto, "La instrucción parece muy corta; agregá contexto o alcance"),
        ("Sin negaciones como punto de partida", not tiene_no, "Reescribí usando lo que SÍ querés en vez de lo que no querés"),
        ("Una tarea a la vez",           not tiene_y_ademas, "Separar en dos prompts si hay múltiples tareas"),
    ]

    for nombre, ok, consejo in checks:
        estado = "✓" if ok else "✗"
        print(f"  {estado}  {nombre}")
        if not ok:
            print(f"       → {consejo}")

    print("─" * 55)


# Probamos con un prompt claramente deficiente
diagnosticar_prompt("Habla de abogacía.")

───────────────────────────────────────────────────────
DIAGNÓSTICO DEL PROMPT
───────────────────────────────────────────────────────
Texto: Habla de abogacía.

  ✗  Tiene rol explícito
       → Agregá 'Sos un...' al comienzo
  ✗  Especifica el formato
       → Indicá cómo querés la respuesta: lista, tabla, párrafo...
  ✗  Instrucción suficientemente larga
       → La instrucción parece muy corta; agregá contexto o alcance
  ✓  Sin negaciones como punto de partida
  ✓  Una tarea a la vez
───────────────────────────────────────────────────────


In [8]:
# Ahora con un prompt mejor estructurado
diagnosticar_prompt(
    "Sos un especialista en marketing digital. "
    "Listá 4 estrategias para aumentar seguidores en Instagram para una marca de ropa sustentable."
)

───────────────────────────────────────────────────────
DIAGNÓSTICO DEL PROMPT
───────────────────────────────────────────────────────
Texto: Sos un especialista en marketing digital. Listá 4 estrategias para aumentar segu...

  ✓  Tiene rol explícito
  ✓  Especifica el formato
  ✓  Instrucción suficientemente larga
  ✓  Sin negaciones como punto de partida
  ✓  Una tarea a la vez
───────────────────────────────────────────────────────


---
## 4. Reformulación sistemática

Para cada anti-patrón hay una reformulación directa. Vamos a ver los cinco en acción comparando la versión rota con la versión corregida.

In [9]:
# ─── Anti-patrón 1: instrucción vaga ─────────────────────────────────────────
roto    = "Escribí algo sobre liderazgo."
corregido = "Sos un coach ejecutivo. Escribí 3 principios de liderazgo aplicables a equipos remotos, en formato de lista numerada, máximo 2 líneas por punto."

print("ROTO:\n", "Prompt:",roto)
print(llamar_llm(roto, max_tokens=100))
print("\n" + "----" * 20)
print()
print("CORREGIDO: \n", "Prompt:", corregido[:60], "...")
print(llamar_llm(corregido, max_tokens=120))

ROTO:
 Prompt: Escribí algo sobre liderazgo.
**El Liderazgo: Un Arte que Impulsa el Éxito**

El liderazgo es una habilidad fundamental para cualquier persona que desee alcanzar sus objetivos y lograr el éxito en su carrera o en la vida personal. Un líder efectivo tiene la capacidad de inspirar, motivar y guiar a otros hacia un propósito común.

**Características del Liderazgo Efectivo**

Un líder exitoso debe poseer las siguientes características:

--------------------------------------------------------------------------------

CORREGIDO: 
 Prompt: Sos un coach ejecutivo. Escribí 3 principios de liderazgo ap ...
Excelente pregunta! Como coach ejecutivo, aquí te presento 3 principios de liderazgo relevantes para equipos remotos:

1. **Clareza y comunicación efectiva**: Establece una línea de comunicación clara y estable entre todos los miembros del equipo, fomentando la transparencia y la retroalimentación. Asegúrate de que cada miembro se sienta escuchado y valorado.

2. **Flexibilida

In [11]:
# ─── Anti-patrón 2: sin rol ──────────────────────────────────────────────
roto    = "Escribí un mensaje para rechazar a un candidato después de una entrevista."
corregido = """Sos un reclutador experimentado en una empresa de tecnología, conocido por tu empatía.
Escribí un mensaje de correo electrónico (máximo 3 párrafos) para rechazar a un candidato después de una
entrevista, manteniendo un tono profesional, alentador y humano."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=200))
print("----" * 20)
print()
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=200))

ROTO:
¡Claro! Aquí te dejo un ejemplo de mensaje que puedes enviar a un candidato que ha sido rechazado después de una entrevista:

"Estimado/a [Nombre del candidato],

Queremos agradecerte por tomar el tiempo para reunirse con nosotros para la entrevista de [posición/vacante] en [nombre de la empresa]. Aunque hemos considerado tu perfil y habilidades, después de una cuidadosa evaluación, hemos decidido avanzar con otro candidato que se ajuste mejor a nuestros requisitos específicos.

Queremos dejar constancia de que has demostrado un interés y compromiso genuinos por el puesto, y queremos asegurarte que tu experiencia y habilidades son valiosas en el mercado laboral actual. Sin embargo, debido a las circunstancias actuales, no podemos ofrecerte la oportunidad de unirse a nuestro equipo.

Agradecemos nuev
--------------------------------------------------------------------------------

CORREGIDO:
Estimado/a \[Nombre del candidato],

Quería tomar un momento para expresar mi más sincero 

In [12]:
# ─── Anti-patrón 3: sin formato ──────────────────────────────────────────
roto    = "Decime los ingredientes para hacer milanesas."
corregido = """Listá los ingredientes para hacer milanesas.
Formato requerido: una lista con viñetas (bullet points), donde cada punto indique cantidad y nombre del ingrediente."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=150))
print("----" * 20)
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=150))

ROTO:
¡Claro! Los ingredientes básicos para hacer milanesas son:

**Ingredientes**

* 4-6 patitas de ternera o pollo, sin hueso ni piel
* 1 taza de harina all-purpose
* 1 cucharadita de sal
* 1/2 cucharadita de pimienta negra molida
* 1 huevo grande, batido
* 1 taza de pan rallado o aros de pan para hornear
* Aceite vegetal o aceite de oliva para freír
* Opcional: lechuga, tomate y salsa para acompañar

**Opcional:** Puedes agregar otros
--------------------------------------------------------------------------------
CORREGIDO:
¡Claro! Aquí tienes la lista de ingredientes necesarios para hacer milanesas:

*   4-6 cortes de pollo (pechuga o muslo)
*   1 taza de harina
*   2 cucharadas de perejil picado
*   1/2 cucharadita de sal
*   1/4 cucharadita de pimienta negra
*   2 huevos
*   1 taza de pan rallado (opcional)
*   Aceite vegetal para freír


In [ ]:
# ─── Anti-patrón 4: múltiples tareas en un prompt ────────────────────────────
roto = """Explicá qué es el machine learning, también describí sus aplicaciones,
y además dame un ejemplo de código en Python y una lista de libros recomendados."""

# En vez de un prompt gigante, partimos en dos tareas enfocadas
tarea_1 = """Sos un docente universitario. Explicá qué es el machine learning
en 3 oraciones simples, sin código, para alguien sin experiencia técnica."""

tarea_2 = """Sos un docente universitario. Listá 3 aplicaciones reales del machine learning
en empresas. Una línea por aplicación."""

print("ROTO — todo junto:")
print(llamar_llm(roto, max_tokens=200))
print("----" * 20)
print("CORREGIDO — tarea 1:")
print(llamar_llm(tarea_1, max_tokens=120))
print("----" * 20)
print("CORREGIDO — tarea 2:")
print(llamar_llm(tarea_2, max_tokens=120))

In [ ]:
# ─── Anti-patrón 5: negaciones sin alternativa ───────────────────────────────
roto      = "Explicá inteligencia artificial. No uses tecnicismos y no seas aburrido."
corregido = """Explicá inteligencia artificial usando una analogía cotidiana.
Tono: conversacional y entusiasta. Máximo 3 oraciones."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=120))
print("----" * 20)
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=120))

---
---
## 5. Actividad: corregir prompts rotos

Cada celda tiene un prompt con un anti-patrón identificado. Tu tarea es reescribir el prompt, ejecutarlo y observar la diferencia.

#### Ejercicio A — Anti-patrón: sin rol ni formato 

In [ ]:
# ─── Ejercicio A — Anti-patrón: sin rol ni formato ───────────────────────────
prompt_roto_a = "Decime cómo mejorar mis presentaciones."

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_a, max_tokens=150))
print()

In [ ]:
# Ejercicio A: Reescribí el prompt con rol, instrucción clara y formato definido
mi_correccion_a = """..."""

print("RESPUESTA CON TU CORRECCIÓN:")
print(llamar_llm(mi_correccion_a, max_tokens=150))

---
#### Ejercicio B — Anti-patrón: negación sin alternativa

In [ ]:
# ─── Ejercicio B — Anti-patrón: negación sin alternativa ─────────────────────
prompt_roto_b = "Escribí un email de ventas. No lo hagas genérico, no uses frases cliché."

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_b, max_tokens=150))
print()

In [ ]:
# Ejercicio B: Reescribí el prompt describiendo lo que SÍ querés
mi_correccion_b = """..."""

print("RESPUESTA CON TU CORRECCIÓN:")
print(llamar_llm(mi_correccion_b, max_tokens=150))

---
#### Ejercicio C: Separar en dos prompts enfocados y ejecutar ambos

In [ ]:
# ─── Ejercicio C — Anti-patrón: múltiples tareas ─────────────────────────────
prompt_roto_c = """Explicá qué es una base de datos relacional, también comparala con
NoSQL, y dame cuándo usar cada una y un ejemplo de empresa real para cada caso."""

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_c, max_tokens=200))
print()

In [ ]:
# Ejercicio C: Separar en dos prompts enfocados y ejecutar ambos
mi_prompt_c1 = """..."""
mi_prompt_c2 = """..."""

print("TAREA 1:")
print(llamar_llm(mi_prompt_c1, max_tokens=130))
print()
print("TAREA 2:")
print(llamar_llm(mi_prompt_c2, max_tokens=130))

---
#### Ejercicios D y E: Diagnóstico y corrección de tus propios prompts

In [ ]:
# ─── Ejercicio D y E — Dos prompts a tu elección ─────────────────────────────
# TODO: Escribí dos prompts rotos propios (de tu trabajo o de un caso que conozcas)
# Identificá el anti-patrón y corregílos.

prompt_roto_d = """..."""
# Anti-patrón detectado: ...
prompt_corregido_d = """..."""

prompt_roto_e = """..."""
# Anti-patrón detectado: ...
prompt_corregido_e = """..."""

for nombre, roto, corregido in [
    ("D", prompt_roto_d, prompt_corregido_d),
    ("E", prompt_roto_e, prompt_corregido_e),
]:
    print(f"=== Ejercicio {nombre} ===")
    print("Roto:")
    print(llamar_llm(roto, max_tokens=130))
    print("Corregido:")
    print(llamar_llm(corregido, max_tokens=130))
    print()